# 2. 处理文本数据
本章学习如何为训练大语言模型准备输入文本。这涉及到将文本分割为独立的单词词元和子词词元，然后将其编码为大语言模型所使用的向量表示。还将了解到高级的分词技术，比如字节对编码（byte pair encoding, BPE），这是一种在 GPT 等流行的大语言模型中广泛使用的方法。最后，本章将实现一种采样和数据加载策略，来生成训练大语言模型所需的输入-输出对。

## 2.1 理解词嵌入
大语言模型无法直接处理原始文本，因为原始文本数据是离散的，因此无法直接使用原始文本数据来执行神经网络所需的数学运算。因此，需要一种将单词表示为连续值的向量格式的方法。

将数据转换为向量格式的过程称为**嵌入（embedding）**。嵌入的本质是将离散对象（如单词、图像甚至整个文档）映射到连续向量空间中的点，其主要目的是将非数值的数据转换为神经网络可以处理的格式。
本章设计的嵌入技术是词嵌入。

接下来，将逐步了解为大语言模型准备嵌入向量的过程，包括将文本分割为单词、将单词转换为词元，以及将词元转换为嵌入向量。

## 2.2 文本分词
本部分介绍如何将输入文本分割为独立的词元，这是为大语言模型生成嵌入向量所必需的预处理步骤。词元既可以是单个单词，也可以是包括标点符号在内的特殊字符。

为了训练大语言模型，使用 Edith Wharton 的短篇小说 The Verdict 作为分词文本，该文本可以在维基文库中找到，并存储为 the-verdict.txt 文本文件；也可以在 Github 仓库中找到该文件并使用 Python 代码下载该文件。

In [1]:
import ssl
import os
import certifi
import  urllib.request

# 设置 SSL 使用 certifi 提供的证书路径
os.environ['SSL_CERT_FILE'] = certifi.where()
os.environ['REQUESTS_CA_BUNDLE'] = certifi.where()

url = "https://raw.githubusercontent.com/rasbt/LLMs-from-scratch/refs/heads/main/ch02/01_main-chapter-code/the-verdict.txt"
file_path = "the-verdict.txt"
urllib.request.urlretrieve(url, file_path)

('the-verdict.txt', <http.client.HTTPMessage at 0x106bb6910>)

In [2]:
# 通过 Python 读取短篇小说 The Verdict 作为文本样本
with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()
print("Total number of character:", len(raw_text))
print(raw_text[:99])

Total number of character: 20479
I HAD always thought Jack Gisburn rather a cheap genius--though a good fellow enough--so it was no 


我们的目标是将这篇包含 20479 个字符的短篇小说分割为独立的单词和特殊字符，以便在后续章节中将它们转换为嵌入向量，进而用于大语言模型的训练。

为了获取词元列表，应该如何有效地分割文本？为此，我们将进行一个小练习，使用 Python 的正则表达式库 re 进行说明。（你无须学习或记忆任何正则表达式语法，因为稍后我们将转而使用预构建的分词器。）

以简单文本为例，通过使用 re.split 命令并配合适当的语法，我们可以按照空白字符分割文本：

In [3]:
import re

text = "Hello, world. This, is a test."
result = re.split(r'(\s)', text)
print(result)

['Hello,', ' ', 'world.', ' ', 'This,', ' ', 'is', ' ', 'a', ' ', 'test.']


在大部分情况下，这种简单的分词方法能够将文本分割为单独的单词。但问题在于，一些单词仍然与标点符号相连，而我们希望这些标点符号作为单独的列表项。此外，我们没有将所有文本都转换为小写，因为大写形式有助于大语言模型区分专有名词和普通名词、更好地理解句子结构，并学会正确地生成大写字母。

接下来修改正则表达式，使其在空白字符（\s）、逗号和句号（[,.]）处进行分割：

In [4]:
result = re.split(r'([,.]|\s)', text)
print(result)

['Hello', ',', '', ' ', 'world', '.', '', ' ', 'This', ',', '', ' ', 'is', ' ', 'a', ' ', 'test', '.', '']


一个问题是列表中仍然包含空白字符，可以通过以下方法安全地删除这些冗余字符：

In [5]:
result = [item for item in result if item.strip()]
print(result)

['Hello', ',', 'world', '.', 'This', ',', 'is', 'a', 'test', '.']


我们设计的分词方法在处理简单文本时表现良好。接下来，让我们再修改一下，使其能够处理其他类型的标点符号，比如问号、引号，以及短篇小说 The Verdict 的前 100 个字符中出现的双破折号等特殊字符：

In [6]:
text = "Hello, world. Is this-- a test?"
result = re.split(r'([,.:;?_!"()\']|--|\s)', text)
result = [item for item in result if item.strip()]
print(result)

['Hello', ',', 'world', '.', 'Is', 'this', '--', 'a', 'test', '?']


现在已经构建了一个简易的分词器，让我们将其应用于短篇小说 The Verdict 的全文：

In [7]:
preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', raw_text)
preprocessed = [item for item in preprocessed if item.strip()]
print(len(preprocessed))

4690


上述打印语句的输出是 4690，这是该文本的词元数量（不包括空白字符）。为了快速查看分词效果，可以打印前 30 个词元：

In [8]:
print(preprocessed[:30])

['I', 'HAD', 'always', 'thought', 'Jack', 'Gisburn', 'rather', 'a', 'cheap', 'genius', '--', 'though', 'a', 'good', 'fellow', 'enough', '--', 'so', 'it', 'was', 'no', 'great', 'surprise', 'to', 'me', 'to', 'hear', 'that', ',', 'in']


结果显示，分词器似乎很好地处理了文本，因为所有单词和特殊字符都被整齐地分开了。

## 2.3 将词元转换为词元 ID
这一节学习将词元从 Python 字符串转换为整数表示，以生成词元 ID（token ID）。这一过程是将词元 ID 转换为嵌入向量前的必经步骤。

为了将之前生成的词元映射到词元 ID，首先需要构建一张词汇表。这张表定义了如何将每一个唯一的单词和特殊字符映射到唯一的整数。具体来说，构建词汇表的过程，首先是将训练集中的全部文本分词成独立的词元；然后将这些词元按字母顺序进行排序，并删除重复的词元；接下来将唯一的词元聚合到一张词汇表中，该词汇表定义了每个唯一的词元到唯一的整数值的映射。

我们现在已经完成了对短篇小说 The Verdict 的分词，并将结果存储在名为 preprocessed 的 Python 变量中。接下来，将创建一个包含所有唯一词元的列表，并将它们按照字母顺序排列，以确定词汇表的大小：

In [9]:
all_words = sorted(set(preprocessed))
vocab_size = len(all_words)
print(vocab_size)

1130


随后，我们创建词汇表，并打印该词汇表的前 51 个条目作为示例。

In [10]:
vocab = {token: integer for integer, token in enumerate(all_words)}
for i, item in enumerate(vocab.items()):
    print(item)
    if i >= 50:
        break

('!', 0)
('"', 1)
("'", 2)
('(', 3)
(')', 4)
(',', 5)
('--', 6)
('.', 7)
(':', 8)
(';', 9)
('?', 10)
('A', 11)
('Ah', 12)
('Among', 13)
('And', 14)
('Are', 15)
('Arrt', 16)
('As', 17)
('At', 18)
('Be', 19)
('Begin', 20)
('Burlington', 21)
('But', 22)
('By', 23)
('Carlo', 24)
('Chicago', 25)
('Claude', 26)
('Come', 27)
('Croft', 28)
('Destroyed', 29)
('Devonshire', 30)
('Don', 31)
('Dubarry', 32)
('Emperors', 33)
('Florence', 34)
('For', 35)
('Gallery', 36)
('Gideon', 37)
('Gisburn', 38)
('Gisburns', 39)
('Grafton', 40)
('Greek', 41)
('Grindle', 42)
('Grindles', 43)
('HAD', 44)
('Had', 45)
('Hang', 46)
('Has', 47)
('He', 48)
('Her', 49)
('Hermia', 50)


如上所示，字典包含着许多独立的词元，它们均与唯一的整数标签相关联。

下一步是使用这张词汇表将新文本转换为词元 ID。

为了将大语言模型的输出从数值形式转换为文本，还需要一种将词元 ID 转换为文本的方法。为此，可以创建逆向词汇表，将词元 ID 映射回它们对应的文本词元。

我们在 Python 中实现一个完整的分词器类。此类包含一个用于将文本分词的 encode 方法，并通过词汇表将字符串映射为整数，以生成词元 ID。此外，我们还将实现一个 decode 方法，执行从整数到字符串的反向映射，将词元 ID 还原为文本。

In [11]:
# 实现一个简易文本分词器
class SimpleTokenizerV1:
    def __init__(self, vocab):
        self.str_to_int = vocab # 将词汇表作为类属性存储，以便在 encode 方法和 decode 方法中访问
        self.int_to_str = {i:s for s,i in vocab.items()} # 创建逆向词汇表，将词元 ID 映射回原始文本词元
    
    def encode(self, text):
        """处理输入文本，将其转换为词元 ID"""
        preprocessed = re.split(r'([,.?_!"()\']|--|\s)', text)
        preprocessed = [
            item.strip() for item in preprocessed if item.strip()
        ]
        ids = [self.str_to_int[s] for s in preprocessed]
        return ids
    
    def decode(self, ids):
        """将词元 ID 转换为文本"""
        text = " ".join([self.int_to_str[i] for i in ids])
        text = re.sub(r'\s+([,.?!"()\'])', r'\1', text) # 移除特定标点符号前的空格
        return text

使用 SimpleTokenizerV1 类，现在可以利用已有的词汇表实例化新的分词器对象，然后再使用这些分词器对象对文本进行编码和解码。

接下来，创建一个 SimpleTokenizerV1 类的分词器实例对象，并将其应用于短篇小说 The Verdict 中的一段文本。

In [12]:
tokenizer = SimpleTokenizerV1(vocab)
text  = """"It's the last he painted, you know,"
        Mrs. Gisburn said with pardonable pride."""
ids = tokenizer.encode(text)
print(ids)

[1, 56, 2, 850, 988, 602, 533, 746, 5, 1126, 596, 5, 1, 67, 7, 38, 851, 1108, 754, 793, 7]


接下来，试试 decode 方法能否将这些词元 ID 还原为文本：

In [13]:
print(tokenizer.decode(ids))

" It' s the last he painted, you know," Mrs. Gisburn said with pardonable pride.


现在，将该分词器应用于训练集之外的新样本：

In [ ]:
text = "Hello, do you like tea?"
# print(tokenizer.encode(text)) # 因为没有'Hello'而导致报错

报错原因在于，“Hello”这一单词并未在小说 The Verdict 中出现，因此没有被收录到词汇表中。这一现象凸显了在处理大语言模型时，使用规模更大且更多样化的训练集来扩展词汇表的必要性。

## 2.4 引入特殊上下文词元
为了处理未知的单词，需要对分词器进行修改。接下来，我们将探讨如何通过引入特殊上下文词元，来增强模型对上下文和其他相关信息的理解。这些特殊词元可能包括用于标识未知词汇和文档边界的词元。现在，我们修改 SimpleTokenizerV1，以支持两个新词元——<|unk|>和<|endoftext|>。<|unk|>词元用来表示没有被包含在词汇表中的新词和未知词，<|endoftext|>词元用来分隔两个不相关的文本来源。

在处理多个独立的文本源时，我们在这些文本之间插入<|endoftext|>词元，这些<|endoftext|>词元作为标记，可以指示出特定文本片段的开始或结束，从而有助于大语言模型更有效地处理和理解文本。

现在，将这两个特殊词元<|unk|>和<|endoftext|>添加到词汇表中：

In [14]:
all_tokens = sorted(list(set(preprocessed)))
all_tokens.extend(["<|endoftext|>", "<|unk|>"])
vocab = {token:integer for integer, token in enumerate(all_tokens)}

print(len(vocab.items()))

1132


更新后的词汇表大小为 1132，更新前为 1130。为了进行快速验证，打印新的词汇表的最后 5 个条目：

In [15]:
for i, item in enumerate(list(vocab.items())[-5:]):
    print(item)

('younger', 1127)
('your', 1128)
('yourself', 1129)
('<|endoftext|>', 1130)
('<|unk|>', 1131)


In [16]:
# 基于 SimpleTokenizerV1，调整为 SimpleTokenizerV2
class SimpleTokenizerV2:
    def __init__(self, vocab):
        self.str_to_int = vocab
        self.int_to_str = {i:s for s,i in vocab.items()}
    
    def encode(self, text):
        preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', text)
        preprocessed = [
            item.strip() for item in preprocessed if item.strip()
        ]
        preprocessed = [item if item in self.str_to_int
                             else "<|unk|>" for item in preprocessed]
        ids = [self.str_to_int[s] for s in preprocessed]
        return ids
    
    def decode(self, ids):
        text = " ".join([self.int_to_str[i] for i in ids])
        text = re.sub(r'\s+([,.:;?!"()\'])', r'\1', text)
        return text

现在来实际测试一下这个新的分词器。为此，我们将使用一个简答的文本样本，该样本由两个独立且无关的句子拼接而成：

In [17]:
text1 = "Hello, do you like tea?"
text2 = "In the sunlit terraces of the palace."
text = " <|endoftext|> ".join((text1, text2))
print(text)

Hello, do you like tea? <|endoftext|> In the sunlit terraces of the palace.


In [18]:
# 接下来，使用 SimpleTokenizerV2 对文本样本进行分词
tokenizer = SimpleTokenizerV2(vocab)
print(tokenizer.encode(text))

[1131, 5, 355, 1126, 628, 975, 10, 1130, 55, 988, 956, 984, 722, 988, 1131, 7]


In [19]:
# 进行反词元化处理，检查分词器有效性
print(tokenizer.decode(tokenizer.encode(text)))

<|unk|>, do you like tea? <|endoftext|> In the sunlit terraces of the <|unk|>.


在实际训练 GPT 模型时，仅使用了 <|endoftext|> 词元，并未使用 <|unk|> 词元，而是使用 BPE 分词器将单词拆解为子词单元。

## 2.5 BPE


这一部分将介绍一种基于 BPE 概念的更复杂的分词方案。BPE 分词器用于训练大语言模型，比如 GPT-2、GPT-3 和 ChatGPT 的原始模型。
由于 BPE 的实现相对复杂，因此我们使用现有的 Python 开源库 tiktoken，它基于 Rust 的源代码非常高效地实现了 BPE 算法。可以使用`pip install tiktoken`安装 tiktoken 库。

In [20]:
# 检查 tiktoken 库的版本
from importlib.metadata import version
import tiktoken

print("tiktoken version:", version("tiktoken"))

tiktoken version: 0.7.0


安装后，可以使用以下方式实例化 tiktoken 中的 BPE 分词器：

In [21]:
tokenizer = tiktoken.get_encoding("gpt2")

这个分词器与前面通过 encode 方法实现的 SimpleTokenizerV2 用法相似：

In [22]:
text = (
    "Hello, do you like tea? <|endoftext|> In the sunlit terraces"
    " of someunknownPlace."
)
integers = tokenizer.encode(text, allowed_special={"<|endoftext|>"})
print(integers)

[15496, 11, 466, 345, 588, 8887, 30, 220, 50256, 554, 262, 4252, 18250, 8812, 2114, 286, 617, 34680, 27271, 13]


In [23]:
# 使用 decode 方法将词元 ID 转换回文本
strings = tokenizer.decode(integers)
print(strings)

Hello, do you like tea? <|endoftext|> In the sunlit terraces of someunknownPlace.


通过分析上述词元 ID 和解码后的文本，我们可以得出两个重要的观察结果：
- <|endoftext|> 词元被分配了一个较大的词元 ID，即 50256。事实上，用于训练 GPT-2、GPT-3 和 ChatGPT 中使用的原始模型的 BPE 分词器的词汇总量为 50257，这意味着 <|endoftext|> 被分配了最大的词元 ID。
- BPE 分词器可以正确地编码和解码未知单词，比如“someunknownPlace”。BPE 分词器是如何做到在不使用 <|unk|> 词元的前提下处理任何位置词汇的呢？
BPE 算法的原理是将不在预定义词汇表中的单词分解为更下的子词单元甚至单个字符，从而能够处理词汇表之外的单词。因此，得益于 BPE 算法，如果分词器在分词过程中遇到不熟悉的单词，它可以将其表示为子词词元或字符序列。
因为 BPE 分词器会将未知单词分解为子词和单个字符，因此 BPE 分词器可以解析任何单词，而无须使用特殊词元 <|unk|> 来替换未知单词。
将未知单词分解为单个字符的能力确保了分词器以及用其训练的大语言模型能够处理任何文本，即使文本中包含训练数据中不存在的单词。

In [24]:
# 练习 2.1
tokenizer = tiktoken.get_encoding("gpt2")
text = "Akwirw ier"
integers = tokenizer.encode(text)
print(integers)
for i in integers:
    print(f"{i}:{tokenizer.decode(list((i,)))}")
strings = tokenizer.decode(integers)
print(strings)

[33901, 86, 343, 86, 220, 959]
33901:Ak
86:w
343:ir
86:w
220: 
959:ier
Akwirw ier


对 BPE 的详细讲解和实现超出了本书的范围，但简单来说，BPE 通过将频繁出现的字符合并为子词，再将频繁出现的子词合并为单词，来迭代地构建词汇表。具体来说，BPE 首先将所有单个字符（如a、b等）添加到词汇表中。然后，它会将频繁同时出现的字符组合合并为子词。例如，d 和 e可以合并为 de，这是“define”、“depend”、“made”、“hidden”等许多英语单词中的常见组合。字符和子词的合并由一个频率阈值决定。

## 2.6 使用滑动窗口进行数据采样


为了生成大语言模型的嵌入向量，接下来的步骤是生成用于训练模型的输入-输出对。这些输入-输出对是什么样的呢？

给定一个文本样本，我们从中提取子样本，作为输入块提供给大语言模型。在训练过程中，模型的任务是预测输入块之后的下一个词，我们会屏蔽目标词之后的所有单词。必须说明的是，在大语言模型处理文本之前，文本会经过分词处理，但为了更清晰地说明，这里省略了分词步骤。

让我们实现一个数据加载器，使用滑动窗口（sliding window）方法从训练数据集中提取输入-输出对。首先，使用 BPE 分词器对短篇小说 The Verdict 的全文进行分词：

In [25]:
with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

enc_text = tokenizer.encode(raw_text)
print(len(enc_text))

5145


执行上述代码将返回 5145，这是应用 BPE 分词器后训练集中的词元总数。

接下来，从数据集中移除前 50 个词元以便于演示，这样做会使得在后续步骤中产生一个更有趣一些的文本段落：

In [26]:
enc_sample = enc_text[50:]

创建下一个单词预测任务的输入-输出对的一种简单且直观的方法是定义两个变量：$x$和$y$。变量$x$用于存储输入的词元，变量$y$用于存储由$x$的每个输入词元右移一个位置所得的目标词元：

In [27]:
context_size = 4 # 上下文大小决定了输入中包含多少词元
x = enc_sample[:context_size]
y = enc_sample[1:context_size+1]
print(f"x: {x}")
print(f"y:      {y}")

x: [290, 4920, 2241, 287]
y:      [4920, 2241, 287, 257]


通过处理输入及其相应的目标（将输入右移一个位置），可以创建下一个单词预测任务：

In [28]:
for i in range(1, context_size+1):
    context = enc_sample[:i]
    desired = enc_sample[i]
    print(context, "---->", desired)

[290] ----> 4920
[290, 4920] ----> 2241
[290, 4920, 2241] ----> 287
[290, 4920, 2241, 287] ----> 257


箭头（---->）左侧的内容表示大语言模型接收的输入，箭头右侧的词元 ID 则代表大语言模型应该预测的目标词元 ID。为了更直观地展示这一过程，让我们重用前面的代码，但这一次将词元 ID 转换为文本形式：

In [29]:
for i in range(1, context_size+1):
    context = enc_sample[:i]
    desired = enc_sample[i]
    print(tokenizer.decode(context), "---->", tokenizer.decode([desired]))

 and ---->  established
 and established ---->  himself
 and established himself ---->  in
 and established himself in ---->  a


这样，可用于大语言模型训练的输入-输出对就创建好了。

在将词元转换为嵌入向量前，还需要完成最后一项任务：实现一个高效的数据加载器（data loader）。这个数据加载器会遍历输入数据集，并将输入和目标以 PyTorch 张量的形式返回，这些 PyTorch 张量可以被视为多维数组。具体来说，我们的目标是返回两个张量：
- 一个包含大语言模型所见的文本输入的输入张量
- 一个包含大语言模型需要预测的目标词元的目标张量

为了实现高效的数据加载器，我们将使用 PyTorch 内置的 Dataset 类和 DataLoader 类。

In [30]:
# 一个用于批处理输入和目标的数据集
import torch
from torch.utils.data import Dataset, DataLoader

class GPTDatasetV1(Dataset):
    def __init__(self, txt, tokenizer, max_length, stride):
        self.input_ids = []
        self.target_ids = []

        token_ids = tokenizer.encode(txt)

        # 使用滑动窗口将文本划分为长度为 max_length 的重叠序列
        for i in range(0, len(token_ids) - max_length, stride):
            input_chunk = token_ids[i:i+max_length]
            target_chunk = token_ids[i+1:i+1+max_length]
            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))
        
    def __len__(self):
        """返回数据集的总行数"""
        return len(self.input_ids)
    
    def __getitem__(self, idx):
        """返回数据集的指定行"""
        return self.input_ids[idx], self.target_ids[idx]

/Users/wyc/Desktop/build_a_llm_from_scratch/.venv/lib/python3.11/site-packages/torch/_subclasses/functional_tensor.py:258: UserWarning: Failed to initialize NumPy: No module named 'numpy' (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/torch/csrc/utils/tensor_numpy.cpp:84.)
  cpu = _conversion_method_template(device=torch.device("cpu"))


GPTDatasetV1 类继承自 PyTorch 的 Dataset 类，并定义了如何从数据集中提取单行数据。每行数据包含多个词元 ID（数量由 max_length 参数决定），这些词元 ID 被分配给 input_chunk 张量，而 target_chunk 张量包含相应的目标词元 ID。

接下来，讲解当数据集与 PyTorch 的 DataLoader 结合使用时返回的数据是什么样的。

In [31]:
# 用于批量生成输入-目标对的数据加载器
def create_dataloader_v1(txt, batch_size=4, max_length=256,
                         stride=128, shuffle=True, drop_last=True,
                         num_workers=0):
    tokenizer = tiktoken.get_encoding("gpt2") # 初始化分词器
    dataset = GPTDatasetV1(txt, tokenizer, max_length, stride) # 创建数据集
    dataloder = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        drop_last=drop_last, # 如果 drop_last 为 True 且批次大小小于指定的 batch_size，则会删除最后一批，以防止在训练期间出现损失剧增
        num_workers=num_workers # 用于预处理的 CPU 进程数
    )
    
    return dataloder

让我们用批次大小为 1 的 DataLoader 对上下文长度为 4 的大语言模型进行测试，以便直观理解 GPTDatasetV1 类和 create_dataloder_v1 函数是如何协同工作的：

In [32]:
with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

dataloader = create_dataloader_v1(raw_text, batch_size=1, max_length=4, stride=1, shuffle=False)
data_iter = iter(dataloader) # 将 dataloader 转换为 Python 迭代器，以通过 Python 内置的 next() 函数获取下一个条目
first_batch = next(data_iter)
print(first_batch)

[tensor([[  40,  367, 2885, 1464]]), tensor([[ 367, 2885, 1464, 1807]])]


变量 first_batch 包含两个张量：第一个张量存储输入词元 ID，第二个张量存储目标词元 ID。由于 max_length 被设置为 4，因此这两个张量各自包含 4 个词元 ID。需要注意的是，为简单起见，我们将输入大小设置为 4，但这个数值是非常小的。在实际训练大语言模型时，输入大小通常不小于 256。

为了说明`stride=1`的含义，需要从数据集中获取另一批数据：

In [33]:
second_batch = next(data_iter)
print(second_batch)

[tensor([[ 367, 2885, 1464, 1807]]), tensor([[2885, 1464, 1807, 3619]])]


如果将第一批数据与第二批数据进行比较，可以发现第二批数据的词元 ID 相对于第一批整体左移了一个位置。步幅（stride）决定了批次之间输入的位移量，模拟了滑动窗口方法。通过在文本上滑动输入窗口来从输入数据集中生成多个批次的数据。如果步幅设置为 1，那么在创建下一个批次时，输入窗口向前移动一个位置；如果步幅与输入窗口大小相等，则可以避免批次之间的重叠。

In [34]:
# 练习 2.2：具有不同步幅和上下文长度的数据加载器
dataloader = create_dataloader_v1(raw_text, batch_size=1, max_length=2, stride=2, shuffle=False)
data_iter = iter(dataloader)
print(next(data_iter))
print(next(data_iter))

print("-------------------------------")

dataloader = create_dataloader_v1(raw_text, batch_size=1, max_length=8, stride=2, shuffle=False)
data_iter = iter(dataloader)
print(next(data_iter))
print(next(data_iter))

[tensor([[ 40, 367]]), tensor([[ 367, 2885]])]
[tensor([[2885, 1464]]), tensor([[1464, 1807]])]
-------------------------------
[tensor([[  40,  367, 2885, 1464, 1807, 3619,  402,  271]]), tensor([[  367,  2885,  1464,  1807,  3619,   402,   271, 10899]])]
[tensor([[ 2885,  1464,  1807,  3619,   402,   271, 10899,  2138]]), tensor([[ 1464,  1807,  3619,   402,   271, 10899,  2138,   257]])]


较小的批次大小（batch_size）会减少训练过程中的内存占用，但同时导致模型更新时产生更多的噪声。正如在常规深度学习训练中一样，批次大小同样是训练大语言模型时需要仔细权衡并尝试调整的超参数。

现在，让我们简单了解一下，如何以大于 1 的批次大小使用数据加载器进行采样：

In [35]:
dataloader = create_dataloader_v1(raw_text, batch_size=8, max_length=4, stride=4, shuffle=False)
data_iter = iter(dataloader)
inputs, targets = next(data_iter)
print("Inputs:\n", inputs)
print("\nTargets:\n", targets)

Inputs:
 tensor([[   40,   367,  2885,  1464],
        [ 1807,  3619,   402,   271],
        [10899,  2138,   257,  7026],
        [15632,   438,  2016,   257],
        [  922,  5891,  1576,   438],
        [  568,   340,   373,   645],
        [ 1049,  5975,   284,   502],
        [  284,  3285,   326,    11]])

Targets:
 tensor([[  367,  2885,  1464,  1807],
        [ 3619,   402,   271, 10899],
        [ 2138,   257,  7026, 15632],
        [  438,  2016,   257,   922],
        [ 5891,  1576,   438,   568],
        [  340,   373,   645,  1049],
        [ 5975,   284,   502,   284],
        [ 3285,   326,    11,   287]])


值得说明的是，我们选择将步幅增加到 4 来充分利用数据集（不会跳过任何一个单词），同时避免不同批次之间的数据重叠，因为过多的重叠可能会增加模型过拟合的风险。

## 2.7 创建词元嵌入


为大语言模型训练准备输入文本的最后一步是将词元 ID 转换为嵌入向量。在初始阶段，必须用随机值初始化这些嵌入权重，这是大语言模型学习的起点。在第五章中，我们将详细探讨如何将嵌入权重作为大语言模型训练的一部分来进行优化。

**大语言模型的输入文本的准备工作包括文本分词、将词元转换为词元 ID，以及将词元 ID 转换为嵌入向量**。本节将利用此前生成的词元 ID 来创建词元嵌入向量。

由于类 GPT 大语言模型是使用反向传播算法（backpropagation algorithm）训练的深度神经网络，因此需要连续的向量表示或嵌入。

让我们通过一个实际的例子来演示词元 ID 转换为嵌入向量的工作原理。假设有 4 个 ID 分别为 2、3、5 和 1 的输入词元：

In [38]:
input_ids = torch.tensor([2, 3, 5, 1])

为简单起见，假设我们有一张仅包含 6 个单词的小型词汇表（而非 BPE 分词器中包含 50257 个单词的词汇表），并且想要创建维度为 3 的嵌入（GPT-3 的嵌入维度是 12288）：

In [39]:
vocab_size = 6
output_dim = 3

可以利用 vocab_size 和 output_dim 在 PyTorch 中实例化一个嵌入层。此外，为了确保结果的可重复性，可以将随机种子设置为 123：

In [40]:
torch.manual_seed(123)
embedding_layer = torch.nn.Embedding(vocab_size, output_dim)
print(embedding_layer.weight)

Parameter containing:
tensor([[ 0.3374, -0.1778, -0.1690],
        [ 0.9178,  1.5810,  1.3010],
        [ 1.2753, -0.2010, -0.1606],
        [-0.4015,  0.9666, -1.1481],
        [-1.1589,  0.3255, -0.6315],
        [-2.8400, -0.7849, -1.4096]], requires_grad=True)


嵌入层的权重矩阵由小的随机值构成。作为模型优化工作的一部分，这些值将在大语言模型训练过程中被优化。此外，**可以观察到，权重矩阵具有 6 行 3 列的结构，其中每一行对应词汇表中的一个词元，每一列对应一个嵌入维度**。

现在将其应用到一个词元 ID 上，以获取嵌入向量：

In [41]:
print(embedding_layer(torch.tensor([3])))

tensor([[-0.4015,  0.9666, -1.1481]], grad_fn=<EmbeddingBackward0>)


通过对比分析可以观察到，词元 ID 为 3 的嵌入向量与嵌入矩阵的第 4 行完全相同。换言之，嵌入层实际上执行的是一种查找操作：根据词元 ID 从嵌入层的权重矩阵中检索出相应的行。

我们已经了解了如何将单个词元 ID 转换为三维嵌入向量。接下来，将这个方法应用到所有 4 个输入 ID 上：

In [42]:
print(embedding_layer(input_ids))

tensor([[ 1.2753, -0.2010, -0.1606],
        [-0.4015,  0.9666, -1.1481],
        [-2.8400, -0.7849, -1.4096],
        [ 0.9178,  1.5810,  1.3010]], grad_fn=<EmbeddingBackward0>)


嵌入层执行查找操作，即从它的权重矩阵中检索与特定词元 ID 对应的嵌入向量。

现在，已经根据词元 ID 创建了嵌入向量。接下来，将对嵌入向量进行细微的调整，以编码词元在文本中的位置信息。

## 2.8 编码单词位置信息
